In [1]:
import math
import random
import torch
import torch.nn as nn
from tqdm import tqdm
from dataset import PAD_IDX, SOS_IDX, EOS_IDX
import sacrebleu
from fix_punctuation import clean_text

In [7]:
'''What teacher forcing probability should I use in epoch X? ; k controls the rate at which teacher forcing probability decreases'''
class InverseSigmoidDecay:
    def __init__(self, k=8.0):
        self.k = k
    def step(self, epoch):
        return self.k / (self.k + math.exp(epoch / self.k))

In [8]:
'''This function converts IDs → clean text.'''
def _decode_ids_to_strings(pred_ids, label_ids, tgt_tokenizer):
    hyps, refs = [], []
    for pred, ref in zip(pred_ids, label_ids):
        '''Convert tensors into Python lists.'''
        pred = pred.tolist()
        ref = ref.tolist()
        '''Remove everything after <EOS> because decoding should stop at the <EOS> token.'''
        if EOS_IDX in pred:
            pred = pred[:pred.index(EOS_IDX)]
        if EOS_IDX in ref:
            ref = ref[:ref.index(EOS_IDX)]
        '''<SOS> , <PAD> ..  should never appear in the decoded sentence'''
        pred = [t for t in pred if t not in (PAD_IDX, SOS_IDX)]
        ref = [t for t in ref if t not in (PAD_IDX, SOS_IDX)]
        '''Convert token IDs into an actual sentence.'''
        hyp_str = tgt_tokenizer.decode(pred, skip_special_tokens=True).strip()
        ref_str = tgt_tokenizer.decode(ref, skip_special_tokens=True).strip()
        '''erform final cleanup (extra spaces, formatting, etc.) before evaluation.'''
        hyps.append(clean_text(hyp_str))
        refs.append(clean_text(ref_str))
    return hyps, refs

In [9]:
'''Run one complete training or validation epoch over all batches in the given DataLoader'''
def _run_epoch(model, dataloader, criterion, optimizer, device, tf_prob: float,
               is_training: bool, tgt_tokenizer=None):
    '''Set the model to training or evaluation mode'''
    if is_training:
        model.train()
    else:
        model.eval()

    '''Accumulate the total training/validation loss across all batches'''
    total_loss = 0.0
    '''Count the total number of non-padding target tokens processed in the epoch'''
    total_tokens = 0
    '''Store all predicted and reference translations for validation metric computation'''
    all_hyps, all_refs = [], []

    '''Enable gradient computation during training; disable it during validation to save memory and computation
       So, ctx is a context manager that switches gradient tracking on for training and off for validation.
    '''
    ctx = torch.enable_grad() if is_training else torch.no_grad()
    
    with ctx:
        '''tqdm is just a library that wraps the iterator and displays a progress bar; desc changes the label on the progress bar.'''
        for batch in tqdm(dataloader, leave=False, desc="Train" if is_training else "Val"):
            '''Source sentence token ids and mask for the current batch'''
            src_ids = batch["src_ids"].to(device)
            src_mask = batch["src_attention_mask"].to(device)
            '''tgt_inputs used for teach forcing , while tgt_labels used to calculate cross-entropy loss'''
            tgt_inputs = batch["tgt_inputs"].to(device)
            tgt_labels = batch["tgt_labels"].to(device)
            tgt_mask = batch["tgt_attention_mask"].to(device)
            B, T = tgt_inputs.shape
            encoder_out = model.encode(src_ids, src_mask)
            hidden = model.init_decoder_hidden(encoder_out)
            '''At the first decoding step, feed the <SOS> token from each target sequence as the initial decoder input'''
            token = tgt_inputs[:, 0]
            logit_slices = []
            '''
                TF = 0.6 means approximately 60 out of every 100 training batches will use teacher forcing, 
                while the remaining 40 batches will decode autoregressively using the model's own predictions.
            '''
            use_teacher_forcing = is_training and random.random() < tf_prob

            for t in range(T):
                '''Forward pass through the decoder for the current time step @t'''
                step_logit, hidden = model.decode_step(token, hidden, encoder_out, src_mask)
                logit_slices.append(step_logit.unsqueeze(1))
                '''Choose the decoder input for the next time step: 
                   use the ground-truth target token with teacher forcing; otherwise feed back the model's highest-probability prediction.
               '''
                if t + 1 < T:
                    if use_teacher_forcing:
                        token = tgt_inputs[:, t + 1]
                    else:
                        token = step_logit.argmax(dim=-1)
                        
            '''T such (B,1,10000) concatenated to logits = (B,T,10000)'''
            logits = torch.cat(logit_slices, dim=1)
            vocab_size = logits.size(-1)
            '''Flatten the batch and sequence dimensions so that CrossEntropyLoss treats every target token prediction as an independent training example'''
            loss = criterion(logits.reshape(-1, vocab_size), tgt_labels.reshape(-1))

            if is_training:
                '''Clear gradients accumulated from the previous training batch'''
                optimizer.zero_grad()
                '''Perform backpropagation to compute gradients of the loss with respect to all trainable model parameters'''
                loss.backward()
                '''Clip gradient norms to prevent exploding gradients and stabilize training'''
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                '''Update the model parameters using the optimizer and the computed gradients'''
                optimizer.step()
                
            '''Count the number of valid (non-padding) target tokens in the current batch'''
            n_tokens = tgt_mask.sum().item()
            '''Accumulate the total cross-entropy loss weighted by the number of valid target tokens'''
            total_loss += loss.item() * n_tokens
            '''Accumulate the total number of valid target tokens processed in the epoch'''
            total_tokens += n_tokens

            if not is_training and tgt_tokenizer is not None:
                '''Convert decoder logits into predicted token IDs and move them to the CPU'''
                preds = logits.argmax(dim=-1).cpu()
                '''Move the ground-truth target token IDs to the CPU'''
                labels = tgt_labels.cpu()
                '''Convert predicted and target token IDs back into text sentences'''
                hyps, refs = _decode_ids_to_strings(preds, labels, tgt_tokenizer)
                '''Accumulate decoded predictions and references for epoch-level evaluation'''
                all_hyps.extend(hyps)
                all_refs.extend(refs)

    '''Compute the average cross-entropy loss over all non-padding target tokens in the epoch'''
    avg_loss = total_loss / max(total_tokens, 1)

    '''During validation, compute and return the BLEU, chrF, and TER evaluation metrics along with the average loss'''
    if not is_training and tgt_tokenizer is not None and len(all_hyps) > 0:
        bleu = sacrebleu.corpus_bleu(all_hyps, [all_refs]).score
        chrf = sacrebleu.corpus_chrf(all_hyps, [all_refs]).score
        ter = sacrebleu.corpus_ter(all_hyps, [all_refs]).score
        return avg_loss, {"bleu": bleu, "chrf": chrf, "ter": ter}

    return avg_loss, None

In [10]:
def train_model(model, train_loader, val_loader, optimizer, device,
                num_epochs: int = 30, save_path: str = "best_model.pth",
                scheduler=None, tf_mode: str = "sigmoid", fixed_tf_ratio: float = 0.6,
                tgt_tokenizer=None):
    '''Use Cross Entropy Loss, ignore padding tokens, and apply label smoothing to improve generalization'''
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
    '''Move the entire model to the selected device (CPU or GPU) for training'''
    model.to(device)
    '''Stores the best validation BLEU score achieved during training'''
    best_val_bleu = -1.0
    '''Counts consecutive epochs with no improvement in validation BLEU'''
    epochs_without_improvement = 0
    '''Reduce the learning rate after 3 epochs without improvement.'''
    scheduler_patience = 3
    '''Stop training if validation BLEU does not improve for 6 consecutive epochs'''
    early_stopping_patience = 6

    '''Teacher forcing scheduler that gradually decreases the teacher forcing probability over epochs'''
    tf_scheduler = InverseSigmoidDecay(k=8.0)

    '''Store training and validation metrics across epochs for later analysis and visualization'''
    history = {
        "train_losses": [], "val_losses": [], "tf_probs": [],
        "val_bleu": [], "val_chrf": [], "val_ter": []
    }

    for epoch in range(num_epochs):
        '''Use inverse sigmoid teacher forcing for Model 4; otherwise use a fixed teacher forcing probability '''
        '''Select the teacher forcing probability for the current epoch'''
        if tf_mode == "sigmoid":
            p_t = tf_scheduler.step(epoch)
        else:
            p_t = fixed_tf_ratio
        '''Run one training epoch over all training batches and return the average training loss'''
        train_loss, _ = _run_epoch(model, train_loader, criterion, optimizer, device,
                                   tf_prob=p_t, is_training=True)
        '''Run one validation epoch without teacher forcing and return the validation loss and evaluation metrics'''
        val_loss, val_metrics = _run_epoch(model, val_loader, criterion, optimizer, device,
                                           tf_prob=0.0, is_training=False, tgt_tokenizer=tgt_tokenizer)
        '''
        NOTE THAT - 
        The target tokenizer is only needed during validation to decode token IDs into text for BLEU, chrF, and TER evaluation
        '''
        '''Store the current epoch's training statistics for later analysis and visualization'''
        history["train_losses"].append(train_loss)
        history["val_losses"].append(val_loss)
        history["tf_probs"].append(p_t)

        metric_str = ""
        current_bleu = 0.0
        if val_metrics:
            '''Stores the current epoch's BLEU score.'''
            current_bleu = val_metrics["bleu"]
            '''Appends all three validation metrics to the history dictionary.'''
            history["val_bleu"].append(current_bleu)
            history["val_chrf"].append(val_metrics["chrf"])
            history["val_ter"].append(val_metrics["ter"])
            metric_str = f" │ BLEU: {val_metrics['bleu']:.2f} │ chrF: {val_metrics['chrf']:.2f} │ TER: {val_metrics['ter']:.2f}"

        '''prints something like for current epoch : Epoch   5/30 │ TF: 0.6000 │ Train: 2.4312 │ Val: 2.7815 │ BLEU: 24.81 │ chrF: 52.63 │ TER: 48.72'''
        print(f"Epoch {epoch+1:3d}/{num_epochs} │ TF: {p_t:.4f} │ Train: {train_loss:.4f} │ Val: {val_loss:.4f}{metric_str}")

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(current_bleu)
            else:
                scheduler.step()
                
        '''If the validation BLEU score improves, the model has performed better'''
        if current_bleu > best_val_bleu:
            
            '''Update the best validation BLEU score and reset the no-improvement counter'''
            best_val_bleu = current_bleu
            epochs_without_improvement = 0
        
            '''Save the current model weights as the best checkpoint'''
            torch.save(model.state_dict(), save_path)
            print(f"  ↳ Saved new best model (val_bleu={current_bleu:.2f})")
        
        else:
            '''Validation BLEU did not improve in this epoch'''
            epochs_without_improvement += 1
            print(f"  ↳ No improvement for {epochs_without_improvement} epochs.")
        
            '''Stop training if the model has not improved for the specified number of consecutive epochs'''
            if epochs_without_improvement >= early_stopping_patience:
                print("  ↳ Early stopping limit reached. Halting training.")
                break

    print(f"\nTraining complete. Best val BLEU: {best_val_bleu:.2f}")
    return history